# Lab 08 — Structured Streaming & Auto Loader

## Traitement incremental en continu

---

### What You Will Learn

1. **Structured Streaming** — treat a Delta table as a live, continuously-appended stream
2. **readStream / writeStream** — the streaming version of read / write
3. **Triggers** — control when processing happens
4. **Watermarking** — handle late-arriving data
5. **Output modes** — append vs complete vs update

### Batch vs Streaming

| | Batch | Streaming |
|---|---|---|
| Reads | `spark.read` | `spark.readStream` |
| Writes | `df.write` | `df.writeStream` |
| Processes | All data at once | New data incrementally |
| Tracking | None (re-reads everything) | Checkpoint (exactly-once) |

---
## Step 0: Setup

Generate streaming-friendly data: a claims table + simulate new batches arriving.

In [ ]:
import random
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col, current_timestamp, window, count, sum as spark_sum, avg, lit
from pyspark.sql import functions as F

# Per-user database
_user = spark.sql("SELECT current_user()").first()[0]
_user_clean = _user.split("@")[0].replace(".", "_").replace("-", "_").lower()
USER_DB = f"training_lab_{_user_clean}"
print(f"Your database: {USER_DB}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {USER_DB}")

# Cleanup previous runs
for t in ["stream_claims_raw", "stream_claims_bronze", "stream_agg_by_type"]:
    spark.sql(f"DROP TABLE IF EXISTS {USER_DB}.{t}")

# Generate initial claims batch (500 records)
claim_types = ["PROPERTY", "LIABILITY", "AUTO", "HEALTH", "MARINE"]
statuses = ["OPEN", "CLOSED", "PENDING"]

data = []
for i in range(1, 501):
    data.append((
        f"CLM-{str(i).zfill(5)}",
        random.choice(claim_types),
        round(random.uniform(1000, 500000), 2),
        random.choice(statuses),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d} {random.randint(0,23):02d}:{random.randint(0,59):02d}:00"
    ))

schema = StructType([
    StructField("claim_id", StringType()),
    StructField("claim_type", StringType()),
    StructField("claim_amount", DoubleType()),
    StructField("status", StringType()),
    StructField("event_time", StringType())
])

spark.createDataFrame(data, schema) \
    .withColumn("event_time", col("event_time").cast("timestamp")) \
    .write.format("delta").mode("overwrite").saveAsTable(f"{USER_DB}.stream_claims_raw")

print(f"Initial batch: {spark.table(f'{USER_DB}.stream_claims_raw').count()} records")

---
## Section 1: readStream — Read a Delta Table as a Stream

Replace `spark.read` with `spark.readStream`. Spark treats the Delta table as a stream and tracks which rows are new using the Delta transaction log.

In [ ]:
# Read the claims table as a stream
# This does NOT read all data immediately — it sets up a streaming query

stream_df = spark.readStream.format("delta").table(f"{USER_DB}.stream_claims_raw")

print(f"Is streaming: {stream_df.isStreaming}")
print(f"Schema:")
stream_df.printSchema()

---
## Section 2: writeStream — Write Stream to Delta

A streaming query needs:
- **format**: where to write (Delta)
- **outputMode**: append (add new rows) / complete (rewrite all) / update (change only updated)
- **checkpointLocation**: tracks progress — enables exactly-once and restart recovery
- **trigger**: when to process — `availableNow=True` processes all then stops

In [ ]:
# Add an ingestion timestamp and write to a Bronze streaming table

bronze_stream = stream_df.withColumn("_ingested_at", current_timestamp())

query = (bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"/tmp/{USER_DB}/stream_ckpt_bronze")
    .trigger(availableNow=True)
    .toTable(f"{USER_DB}.stream_claims_bronze"))

query.awaitTermination()
print(f"Bronze stream complete: {spark.table(f'{USER_DB}.stream_claims_bronze').count()} records")

---
## Section 3: Incremental Processing — Add New Data

The power of streaming: when new rows arrive in the source table, re-running the stream processes **only the new rows** (the checkpoint tracks what was already processed).

In [ ]:
# Simulate new data arriving: append 100 new claims to the source table

new_data = []
for i in range(501, 601):
    new_data.append((
        f"CLM-{str(i).zfill(5)}",
        random.choice(claim_types),
        round(random.uniform(1000, 500000), 2),
        random.choice(statuses),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d} {random.randint(0,23):02d}:{random.randint(0,59):02d}:00"
    ))

spark.createDataFrame(new_data, schema) \
    .withColumn("event_time", col("event_time").cast("timestamp")) \
    .write.format("delta").mode("append").saveAsTable(f"{USER_DB}.stream_claims_raw")

print(f"Source table now has: {spark.table(f'{USER_DB}.stream_claims_raw').count()} records")
print(f"Bronze table still has: {spark.table(f'{USER_DB}.stream_claims_bronze').count()} records")
print("\nThe new 100 records are in the source but NOT yet in Bronze...")

In [ ]:
# Re-run the stream — it processes ONLY the 100 new records (not all 600)
# The checkpoint knows where it left off

stream_df2 = spark.readStream.format("delta").table(f"{USER_DB}.stream_claims_raw")
bronze_stream2 = stream_df2.withColumn("_ingested_at", current_timestamp())

query2 = (bronze_stream2.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"/tmp/{USER_DB}/stream_ckpt_bronze")
    .trigger(availableNow=True)
    .toTable(f"{USER_DB}.stream_claims_bronze"))

query2.awaitTermination()
print(f"Bronze table now has: {spark.table(f'{USER_DB}.stream_claims_bronze').count()} records")
print("Only the 100 new records were processed — exactly-once via checkpoint!")

---
## Section 4: Streaming Aggregation with Watermarking

**Watermarking** tells Spark how long to wait for late-arriving data before finalizing a window.

- `withWatermark("event_time", "1 day")` — wait up to 1 day for late events
- `window("event_time", "7 days")` — aggregate into 7-day buckets
- Output mode must be `complete` or `update` for aggregations

In [ ]:
# Streaming aggregation: total claim amount per claim_type per 30-day window
# Watermark = 1 day — data arriving > 1 day late is dropped

agg_stream = (spark.readStream
    .format("delta")
    .table(f"{USER_DB}.stream_claims_bronze")
    .withWatermark("event_time", "1 day")
    .groupBy(
        window("event_time", "30 days"),
        "claim_type"
    )
    .agg(
        count("*").alias("claim_count"),
        F.round(spark_sum("claim_amount"), 2).alias("total_amount"),
        F.round(avg("claim_amount"), 2).alias("avg_amount")
    ))

# Write aggregation — complete mode rewrites the entire result each time
query_agg = (agg_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", f"/tmp/{USER_DB}/stream_ckpt_agg")
    .trigger(availableNow=True)
    .toTable(f"{USER_DB}.stream_agg_by_type"))

query_agg.awaitTermination()

print("Streaming aggregation complete!")
display(spark.table(f"{USER_DB}.stream_agg_by_type").orderBy("claim_type", "window"))

---
## Section 5: Triggers Explained

| Trigger | Behavior | Use Case |
|---------|----------|----------|
| `trigger(availableNow=True)` | Process all available data, then stop | Scheduled batch-like jobs |
| `trigger(processingTime='10 seconds')` | Process every 10 seconds continuously | Near real-time dashboards |
| `trigger(once=True)` | Process one micro-batch, then stop | One-shot incremental load |
| *(no trigger)* | Process as fast as possible, continuously | Lowest latency streaming |

In this lab we use `availableNow=True` because it works on serverless and stops cleanly.

---
## What You Learned

| Concept | Key API | What It Does |
|---------|---------|-------------|
| Read stream | `spark.readStream.table()` | Treat a Delta table as a live stream |
| Write stream | `df.writeStream.toTable()` | Write streaming results to Delta |
| Checkpoint | `.option("checkpointLocation", path)` | Track progress for exactly-once |
| Incremental | Re-run same stream | Only new rows processed (checkpoint) |
| Watermark | `.withWatermark(col, delay)` | Handle late-arriving data |
| Window | `window(col, duration)` | Time-based aggregation buckets |
| Trigger | `.trigger(availableNow=True)` | Process all then stop |

---
## Cleanup

In [ ]:
# Cleanup (optional)
# for t in ["stream_claims_raw", "stream_claims_bronze", "stream_agg_by_type"]:
#     spark.sql(f"DROP TABLE IF EXISTS {USER_DB}.{t}")
# for p in ["stream_ckpt_bronze", "stream_ckpt_agg"]:
#     try: dbutils.fs.rm(f"/tmp/{USER_DB}/{p}", recurse=True)
#     except: pass
# print("Cleanup complete")